In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException
import time
import pandas as pd

# Path to chromedriver.exe
chrome_driver_path = 'C:/Users/USER/OneDrive/Desktop/chromedriver-win64/chromedriver.exe'  # Update this path

# Set up Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--window-size=1920,1080")  # Set window size

# Set up ChromeDriver service
service = Service(chrome_driver_path)

# Initialize the Chrome WebDriver
driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 60)  # Set the wait time to 60 seconds

def load_page_with_retry(url, retries=3, delay=10):
    """Try to load the page with retries in case of failure."""
    for attempt in range(retries):
        try:
            driver.get(url)
            driver.save_screenshot(f"page_load_attempt_{attempt}.png")
            print(f"Attempt {attempt + 1}: Page loaded successfully.")
            return True
        except TimeoutException:
            print(f"Attempt {attempt + 1}: Page did not load. Retrying in {delay} seconds...")
            time.sleep(delay)
    return False

def get_promoter_details():
    """Extract promoter details including name, PAN number, GSTIN number, and permanent address."""
    try:
        promoter_name_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'td.fw-600')))
        pan_td_element = wait.until(EC.presence_of_element_located((By.XPATH, '//td[text()="PAN No."]')))
        pan_value_element = pan_td_element.find_element(By.XPATH, 'following-sibling::td')
        
        # Clean PAN number text
        pan_number = pan_value_element.text
        if "PAN File" in pan_number:
            pan_number = pan_number.replace("PAN File", "").strip()

        gstin_td_element = wait.until(EC.presence_of_element_located((By.XPATH, '//td[text()="GSTIN No."]')))
        gstin_value_element = gstin_td_element.find_element(By.XPATH, 'following-sibling::td/span[@class="mr-1 fw-600"]')
        
        address_td_element = wait.until(EC.presence_of_element_located((By.XPATH, '//td[text()="Permanent Address"]')))
        address_value_element = address_td_element.find_element(By.XPATH, 'following-sibling::td/span[@class="fw-600"]')
        
        return (promoter_name_element.text if promoter_name_element else "No promoter name found",
                pan_number,
                gstin_value_element.text if gstin_value_element else "No GSTIN number found",
                address_value_element.text if address_value_element else "No permanent address found")
    except StaleElementReferenceException:
        return ("No promoter name found", "No PAN number found", "No GSTIN number found", "No permanent address found")
    except NoSuchElementException:
        return ("No promoter name found", "No PAN number found", "No GSTIN number found", "No permanent address found")

def close_modal():
    """Close the modal window."""
    try:
        close_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @data-dismiss="modal"]')))
        close_button.click()
        print("Modal closed successfully.")
    except TimeoutException:
        print("Failed to close the modal.")
    except NoSuchElementException:
        print("Close button not found.")

def process_rera_number(rera_number):
    """Process the RERA number and extract promoter details."""
    try:
        # Click on the RERA number link
        rera_link = wait.until(EC.element_to_be_clickable((By.XPATH, f'//a[contains(text(), "{rera_number}")]')))
        rera_link.click()

        # Click on "Promoter Details"
        promoter_details_link = wait.until(
            EC.element_to_be_clickable((By.XPATH, '//span[text()="Promoter Details"]'))
        )
        promoter_details_link.click()

        # Extract the promoter details
        name, pan_number, gstin_number, address = get_promoter_details()

        # Return the details
        return {
            'RERA Number': rera_number,
            'Name': name,
            'PAN Number': pan_number,
            'GSTIN Number': gstin_number,
            'Permanent Address': address
        }

    except (TimeoutException, NoSuchElementException) as e:
        print(f"An error occurred: {e}")
        return {
            'RERA Number': rera_number,
            'Name': "Error retrieving name",
            'PAN Number': "Error retrieving PAN",
            'GSTIN Number': "Error retrieving GSTIN",
            'Permanent Address': "Error retrieving address"
        }
    finally:
        # Ensure the modal is closed before proceeding
        close_modal()

try:
    # Open the webpage with retry mechanism
    page_loaded = load_page_with_retry("https://hprera.nic.in/PublicDashboard")
    if not page_loaded:
        raise TimeoutException("Failed to load the page after several attempts.")

    # Click on the "Registered Projects" tab
    registered_projects_tab = wait.until(
        EC.element_to_be_clickable((By.XPATH, '//a[@data-target="#reg-Projects"]'))
    )
    registered_projects_tab.click()

    # Wait for the projects section to be visible
    projects_section = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, '#reg-Projects')))

    all_promoter_details = []

    for i in range(6):
        # Wait for the list of RERA numbers to be visible and get all the links
        rera_links = wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, '#reg-Projects a[title="View Application"]'))
        )

        if i < len(rera_links):
            # Get the RERA number and process it
            rera_number = rera_links[i].text
            print(f"Processing RERA number: {rera_number}")
            
            details = process_rera_number(rera_number)
            all_promoter_details.append(details)

            # Click on the "Registered Projects" tab again to return to the list
            registered_projects_tab = wait.until(
                EC.element_to_be_clickable((By.XPATH, '//a[@data-target="#reg-Projects"]'))
            )
            registered_projects_tab.click()

            # Wait for the next RERA numbers to be visible
            projects_section = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, '#reg-Projects')))
        else:
            print("Not enough RERA numbers found.")
            break

    # Save all details to a single Excel file
    df = pd.DataFrame(all_promoter_details)
    df.to_excel('promoter_details_all.xlsx', index=False, engine='openpyxl')
    print("All details saved to promoter_details_all.xlsx")

finally:
    # Close the browser
    driver.quit()


Attempt 1: Page loaded successfully.
Processing RERA number: RERAHPKUP01180019
Modal closed successfully.
Processing RERA number: RERAHPSOP10170012
Modal closed successfully.
Processing RERA number: HPRERAUNA2022007/P
Modal closed successfully.
Processing RERA number: RERAHPKAP11170013
Modal closed successfully.
Processing RERA number: RERAHPSHP11170015
Modal closed successfully.
Processing RERA number: RERAHPSOP12170016
Modal closed successfully.
All details saved to promoter_details_all.xlsx
